# MQTT with SenML payloads

MQTT moves messages by topic; SenML (RFC 8428) gives sensor payloads a standard shape. IoTCom.Net adapts MQTTnet rather than re-implementing MQTT.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.4.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.4.0-preview.1"

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Adapters.Mqtt;
using IoTCom.Net.Serialization.SenML;

var broker = MqttBroker.Create(18830);
await broker.StartAsync();
var sensor = MqttEndpoint.Create(o => o.UseBroker("127.0.0.1", 18830));
var dashboard = MqttEndpoint.Create(o => o.UseBroker("127.0.0.1", 18830));
await sensor.ConnectAsync();
await dashboard.ConnectAsync();

## Subscribe with wildcards, publish SenML

In [ ]:
using var cts = new CancellationTokenSource(TimeSpan.FromSeconds(3));
var received = Task.Run(async () =>
{
    await foreach (var m in dashboard.SubscribeAsync("plant/+/sensor", cts.Token))
        return SenMLCodec.Resolve(SenMLCodec.ParseJson(m.Payload.Span));
    return null;
});
await Task.Delay(300);

var pack = new SenMLPackBuilder("urn:dev:demo:").At(DateTimeOffset.UtcNow).Add("temperature", 23.4, "Cel").Build();
await sensor.PublishAsync("plant/line1/sensor", SenMLCodec.ToJson(pack));
foreach (var r in (await received)!) Console.WriteLine($"{r.Name} = {r.Value} {r.Unit} @ {r.Time:T}");

## JSON vs CBOR size


In [ ]:
Console.WriteLine($"JSON {SenMLCodec.ToJson(pack).Length} bytes · CBOR {SenMLCodec.ToCbor(pack).Length} bytes");

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*